#clean and transfom data

#read data fome bronze layer

In [0]:
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

import pyspark.sql.functions as F

In [0]:
df=spark.table("workspace.bronze_layer.crm_cust_info")
df.display()

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.display()

In [0]:

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

In [0]:
df.display()

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

In [0]:
df = df.filter(col("customer_id").isNotNull())

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver_layer.crm_customers")

In [0]:
%sql
SELECT * FROM workspace.silver_layer.crm_customers LIMIT 10
     